In [ ]:
#@title Step 1: Set up environment

!git clone -b notebook https://github.com/clami66/resTrain.git
!pip install -q --no-warn-conflicts 'colabfold[alphafold-minus-jax] @ git+https://github.com/sokrypton/ColabFold'
!rm -f /usr/local/lib/python3.*/dist-packages/tensorflow/core/kernels/libtfkernel_sobol_op.so

import os
import glob
import shutil
import importlib_metadata
from sys import path
from pathlib import Path
from string import ascii_uppercase, ascii_lowercase
ascii_upperlower = ascii_uppercase + ascii_lowercase

from google.colab import files
from Bio import SeqIO

import ipywidgets as wg

path.insert(0,'/content/resTrain')

from colabfold.batch import get_msa_and_templates, msa_to_str
from colabfold.utils import DEFAULT_API_SERVER, get_commit
from colabfold.download import download_alphafold_params

from resTrain.run_alphafold import predict_structure
from resTrain.alphafold.data import pipeline, pipeline_multimer

from resTrain.alphafold.data.tools import hmmsearch, jackhmmer
from resTrain.alphafold.data import templates

from resTrain.alphafold.model import model, data, config

In [ ]:


#@title Step 2: Upload input sequences (.fasta)
jobname = "test" #@param {type:"string"}

run_example = False #@param {type:"boolean"}
#@markdown - Check box to use example dimer (CASP target H1142)
#@markdown - Otherwise, run this cell and upload a sequence input followed by a structural template
out_dir = Path(f"{jobname}")
out_dir.mkdir(parents=True, exist_ok=True)
targets = out_dir.joinpath(f"{jobname}.fasta")

if not run_example:
  print("Upload the input sequences as a single fasta file")
  uploaded = files.upload()
  fasta = Path(list(uploaded.keys())[0])
else:
  fasta = Path("/content/resTrain/examples/H1142/H1142.fasta")
  restraints = "/content/resTrain/examples/H1142/restraints.tsv"

_ = shutil.copyfile(fasta, targets)

with open(targets, "r") as f:
  print("Fasta sequences:")
  print(f.read())
  print()

seq2chain = {}
chain_idx = 0
for record in SeqIO.parse(targets, "fasta"):
  if record.seq not in seq2chain:
    seq2chain[record.seq] = [ascii_upperlower[chain_idx]]
  else:
    seq2chain[record.seq].append(ascii_upperlower[chain_idx])
  chain_idx += 1



In [ ]:
#@title Step 3: Set restraints as distances between AA pairs

#@markdown - Run this cell to set the pair distance restraints. These are set as distances between pairs of CB atoms (CA for Glycine)

#@markdown - Select chain and residue number for each AA in the pair and a distance. In the following cell you can select whether the distance is "exact" or "approximate" (i.e. maximum distance)

chains = [ascii_upperlower[i] for i in range(chain_idx)]

def restraint_widget(i):
  chain1 = wg.Dropdown(
      options=chains,
      value=chains[0],
      description='Chain 1:',
      disabled=False,
  )

  chain2 = wg.Dropdown(
      options=chains,
      value=chains[-1],
      description='Chain 2:',
      disabled=False,
  )

  ix1 = wg.IntText(
      value='1',
      description='Res 1:',
      disabled=False,
      layout=wg.Layout(width='200px'),
  )

  ix2 = wg.IntText(
      value='1',
      description='Res 2:',
      disabled=False,
      layout=wg.Layout(width='200px'),
  )

  d = wg.FloatText(
      value='8.0',
      description='Dist.:',
      disabled=False,
      layout=wg.Layout(width='200px'),
  )
  return [wg.HBox([chain1, ix1, chain2, ix2, d])]

vb = wg.VBox(restraint_widget(0))
btn = wg.Button(description = 'Add new restraint')
btn2 = wg.Button(description = 'Delete restraint')

def on_bttn_clicked(b):
    vb.children=tuple(list(vb.children) + restraint_widget(1))

def on_bttn_clicked2(b):
    vb.children=tuple(list(vb.children)[:-1])

btn.on_click(on_bttn_clicked)
btn2.on_click(on_bttn_clicked2)
display(vb, btn, btn2)

In [ ]:
#@title Step 4: Set other parameters
#@markdown #### Inference settings

model_type = "multimer_v2" #@param ["multimer_v2", "multimer_v3", "monomer_ptm"]
model_number = 1 #@param [1, 2, 3, 4, 5]

predictions_per_model = 1 #@param {type:"integer"}
num_recycles = 20 #@param {type:"integer"}
recycle_early_stop_tolerance = 0.5 #@param {type:"number"}
use_dropout = False #@param {type:"boolean"}

#@markdown #### Training settings

approximate_restraints = False #@param {type:"boolean"}
#@markdown - approximate restraints are treated as maximum (rather than exact) distances

optimization_steps = 5 #@param {type:"integer"}
crop_size = 512 #@param {type:"number"}

#@markdown #### MSA settings
msa_mode = "mmseqs2_uniref" #@param ["mmseqs2_uniref", "mmseqs2_uniref_env"]

msa_depth = 256 #@param {type:"integer"}
#@markdown - Templates are always disabled
#@markdown - Multimer MSAs are always unpaired

if msa_depth == "auto": msa_depth = None

# get restraints from previous widget cell
if not run_example:
  restraints = out_dir.joinpath(f"restraints.tsv")
  with open(restraints, "w") as f:
    for child in list(vb.children):
      ch1, r1, ch2, r2, d = (child.children[i].value for i in range(5))
      f.write(f"{ch1}\t{r1}\t{ch2}\t{r2}\t{d}\n")


In [ ]:
#@title Step 5: Get alignments from ColabFold's MSA server

def get_target_data(paths):
    target_sequences = [record.seq for record in SeqIO.parse(paths[0], "fasta")]
    target_chains = (
        [ascii_upperlower[i] for i, s in enumerate(target_sequences)]
    )

    return target_chains, target_sequences


target_chains, target_sequences = get_target_data(
            [str(targets)]
        )

print("Querying ColabFold's MSA server")
msa_lines = None
use_templates = False
custom_template_path = None
pair_mode = "unpaired"
pairing_strategy = "greedy"
host_url = DEFAULT_API_SERVER
version = importlib_metadata.version("colabfold")
commit = get_commit()
if commit:
    version += f" ({commit})"
user_agent = f"colabfold/{version}"

unpaired_msa, paired_msa, query_seqs_unique, query_seqs_cardinality, template_features = get_msa_and_templates(jobname, target_sequences, msa_lines, out_dir, msa_mode, use_templates,
                        custom_template_path, pair_mode, pairing_strategy, host_url, user_agent)

print("Writing out results...")
for sequence, msa in zip(query_seqs_unique, unpaired_msa):
  chains = seq2chain[sequence]
  for chain in chains:
    msa_path = f"msas/{chain}"

    Path(out_dir, msa_path).mkdir(parents=True, exist_ok=True)
    if "multimer" in model_type:
      out_dir.joinpath(f"msas/{chain}/mmseqs2_hits.a3m").write_text(msa)
    else:
      out_dir.joinpath(f"msas/mmseqs2_hits.a3m").write_text(msa)
print("Done.")


In [ ]:
#@title Step 6: Train and predict

data_dir = Path("/content")
version = "v3" if "v3" in model_type else "v2" if "v2" in model_type else ""
model_download_name = f"alphafold2_{model_type}" if "multimer" in model_type else "alphafold2_ptm"

if not glob.glob(f"{data_dir}/params/*{version}*_finished.txt"):
  print("downloading AF parameters...")
  download_alphafold_params(model_download_name, data_dir)

template_searcher = hmmsearch.Hmmsearch(
    binary_path=shutil.which("hmmsearch"),
    hmmbuild_binary_path=shutil.which("hmmbuild"),
    database_path=out_dir.joinpath(f"template_data/pdb_seqres.txt"))

template_featurizer = templates.HmmsearchHitFeaturizer(
    mmcif_dir="/content",
    max_template_date="1900-01-01",
    max_hits=4,
    kalign_binary_path=shutil.which("kalign"),
    release_dates_path=None,
    obsolete_pdbs_path=None)

monomer_data_pipeline = pipeline.DataPipeline(
    mmseqs2_binary_path=Path("mmseqs2"),
    mmseqs2_uniref_database_path="/content",
    mmseqs2_env_database_path=None,
    jackhmmer_binary_path=None,
    hhblits_binary_path=None,
    uniref90_database_path="",
    mgnify_database_path="",
    bfd_database_path="",
    uniref30_database_path="",
    small_bfd_database_path="",
    template_searcher=template_searcher,
    template_featurizer=template_featurizer,
    use_small_bfd=False,
    use_precomputed_msas=True,
    mmseqs2_max_hits=msa_depth,
    restraint_file=restraints if "multimer" not in model_type else None,)

if "multimer" in model_type:
  data_pipeline = pipeline_multimer.DataPipeline(
      monomer_data_pipeline=monomer_data_pipeline,
      jackhmmer_binary_path=shutil.which("jackhmmer"),
      uniprot_database_path=None,
      use_precomputed_msas=True,
      max_uniprot_hits=1,
      separate_homomer_msas=True,
      use_mmseqs2_align=True,
      restraint_file=restraints)
else:
  data_pipeline = monomer_data_pipeline

model_names = [f"model_{model_number}_multimer_{version}" if "multimer" in model_type else f"model_{model_number}_ptm"]

model_runners = {}

for model_name in model_names:
    model_config = config.model_config(model_name)
    model_config.model.num_ensemble_eval = 1
    model_config.model.embeddings_and_evoformer.cross_chain_templates = True
    model_config.model.num_recycle = num_recycles
    model_config.model.global_config.eval_dropout = use_dropout
    model_config.model.recycle_early_stop_tolerance = recycle_early_stop_tolerance

    model_params = data.get_model_haiku_params(
        model_name=model_name, data_dir=str(data_dir))
    model_runner = model.RunModel(model_config, model_params)
    for i in range(predictions_per_model):
      model_runners[f'{model_name}_pred_{i}'] = model_runner

predict_structure(
    fasta_path=targets,
    fasta_name=jobname,
    output_dir_base="/content",
    data_pipeline=data_pipeline,
    model_runners=model_runners,
    benchmark=False,
    random_seed=0,
    restraints=restraints,
    optimization_steps=optimization_steps,
    crop_size=crop_size,
    approximate=approximate_restraints)

In [ ]:
#@title View predictions

#@markdown Restrained pairs are connected by dashed lines

import py3Dmol
import matplotlib.pyplot as plt
from colabfold.colabfold import plot_plddt_legend
from colabfold.colabfold import pymol_color_list, alphabet_list

rank_num = 0 #@param ["0", "1", "2", "3", "4", "5"] {type:"raw"}
color = "chain" #@param ["chain", "lDDT", "rainbow"]
show_sidechains = False #@param {type:"boolean"}
show_mainchains = False #@param {type:"boolean"}

def show_pdb(pdb_file, extension, show_sidechains=False, show_mainchains=False, color="lDDT"):
  view = py3Dmol.view(js='https://3dmol.org/build/3Dmol.js',)
  view.addModel(open(pdb_file,'r').read(), extension)

  if color == "lDDT":
    view.setStyle({'cartoon': {'colorscheme': {'prop':'b','gradient': 'roygb','min':50,'max':90}}})
  elif color == "rainbow":
    view.setStyle({'cartoon': {'color':'spectrum'}})
  elif color == "chain":
    chains = len(target_sequences) + 1
    for n,chain,color in zip(range(chains),alphabet_list,pymol_color_list):
       view.setStyle({'chain':chain},{'cartoon': {'color':color}})

  if show_sidechains:
    BB = ['C','O','N']
    view.addStyle({'and':[{'resn':["GLY","PRO"],'invert':True},{'atom':BB,'invert':True}]},
                        {'stick':{'colorscheme':f"WhiteCarbon",'radius':0.3}})
    view.addStyle({'and':[{'resn':"GLY"},{'atom':'CA'}]},
                        {'sphere':{'colorscheme':f"WhiteCarbon",'radius':0.3}})
    view.addStyle({'and':[{'resn':"PRO"},{'atom':['C','O'],'invert':True}]},
                        {'stick':{'colorscheme':f"WhiteCarbon",'radius':0.3}})
  if show_mainchains:
    BB = ['C','O','N','CA']
    view.addStyle({'atom':BB},{'stick':{'colorscheme':f"WhiteCarbon",'radius':0.3}})

  with open(restraints) as res:
    for restraint in res:
      ch1, r1, ch2, r2, d = restraint.strip().split("\t")
      r1 = int(r1)
      r2 = int(r2)
      view.addResLabels({"chain": ch1, "resi": r1},{"backgroundColor": 0xb0b0b0, "backgroundOpacity": 0.8, "inFront": True})
      view.addResLabels({"chain": ch2, "resi": r2},{"backgroundColor": 0xb0b0b0, "backgroundOpacity": 0.8, "inFront": True})

      view.addCylinder({"start": {"chain": ch1, "resi": r1, "atom": "CA"},
                        "end": {"chain": ch2, "resi": r2, "atom": "CA"},
                        "dashed": True, "color":'#FFFAA0', "fromCap":'round', "toCap":'round'});
  view.zoomTo()
  return view

prediction_pdb = f"{out_dir}/ranked_{rank_num}.pdb"

print("Prediction")
show_pdb(prediction_pdb, "pdb", show_sidechains, show_mainchains, color).show()